# Tools - Ferramentas

Ferramentas são interfaces que um agente, cadeia ou LLM podem usar para interagir com o mundo. Elas combinam algumas coisas:

- O nome da ferramenta
- Uma descrição do que é a ferramenta
- Esquema JSON dos inputs (entradas) da ferramenta
- A função a ser chamada
- Se o resultado de uma ferramenta deve ser retornado diretamente para o usuário

É útil ter todas essas informações porque essas informações podem ser usadas para construir sistemas que tomam ações! O nome, a descrição e o esquema JSON podem ser usados para solicitar ao LLM que ele saiba como especificar qual ação tomar, e então a função a ser chamada é equivalente a tomar essa ação.

Quanto mais simples for a entrada de uma ferramenta, mais fácil será para um LLM conseguir usá-la. Importante ressaltar que o nome, a descrição e o esquema JSON (se utilizados) são todos utilizados no prompt. Portanto, é muito importante que eles sejam claros e descrevam exatamente como a ferramenta deve ser usada. Pode ser necessário alterar o nome padrão, a descrição ou o esquema JSON se o LLM não estiver entendendo como usar a ferramenta.

## Criando tools com o decorator **`@tools`**

In [2]:
from langchain.agents import tool

@tool
def retorna_temperatura_atual(localidade: str) -> str:
	"""Faz busca online de temperatura de uma localidade"""
	return "25ºC"

retorna_temperatura_atual

StructuredTool(name='retorna_temperatura_atual', description='Faz busca online de temperatura de uma localidade', args_schema=<class 'langchain_core.utils.pydantic.retorna_temperatura_atual'>, func=<function retorna_temperatura_atual at 0x10fab6700>)

In [3]:
retorna_temperatura_atual.name

'retorna_temperatura_atual'

In [4]:
retorna_temperatura_atual.description

'Faz busca online de temperatura de uma localidade'

In [5]:
retorna_temperatura_atual.args

{'localidade': {'title': 'Localidade', 'type': 'string'}}

### Descrevendo os argumentos

In [6]:
from langchain.agents import tool
from pydantic import BaseModel, Field

class RetornaTempArgs(BaseModel):
	localidade: str = Field(description="Localidade a ser buscada", examples=["São Paulo", "Porto Alegre"])

@tool(args_schema=RetornaTempArgs)
def retorna_temperatura_atual(localidade: str) -> str:
	"""Faz busca online de temperatura de uma localidade"""
	return "25ºC"

retorna_temperatura_atual

StructuredTool(name='retorna_temperatura_atual', description='Faz busca online de temperatura de uma localidade', args_schema=<class '__main__.RetornaTempArgs'>, func=<function retorna_temperatura_atual at 0x10fb60c20>)

In [7]:
retorna_temperatura_atual.args

{'localidade': {'description': 'Localidade a ser buscada',
  'examples': ['São Paulo', 'Porto Alegre'],
  'title': 'Localidade',
  'type': 'string'}}

### Chamando a tool

In [8]:
retorna_temperatura_atual.invoke({"localidade": "Porto Alegre"})

'25ºC'

## Criando tool com StructuredTool

Outra forma de criar uma tool sem o decorator é utilizando metaclasse StructuredTool do LangChain. As funcionalidades são bem similares, então você pode utilizar um ou outro dependendo da sua preferência.

In [10]:
from langchain.tools import StructuredTool
from pydantic import BaseModel, Field

class RetornaTempArgs(BaseModel):
	localidade: str = Field(description="Localidade a ser buscada", examples=["São Paulo", "Porto Alegre"])

def retorna_temperatura_atual_func(localidade: str) -> str:
	"""Faz busca online de temperatura de uma localidade"""
	return "25ºC"

tool_temp = StructuredTool.from_function(
	func=retorna_temperatura_atual_func,
	name="ToolTemperatura",
	description="Faz busca online de temperatura de uma localidade",
	args_schema=RetornaTempArgs,
	return_direct=True
)

tool_temp

StructuredTool(name='ToolTemperatura', description='Faz busca online de temperatura de uma localidade', args_schema=<class '__main__.RetornaTempArgs'>, return_direct=True, func=<function retorna_temperatura_atual_func at 0x10fb637e0>)

In [11]:
tool_temp.name

'ToolTemperatura'

In [12]:
tool_temp.args

{'localidade': {'description': 'Localidade a ser buscada',
  'examples': ['São Paulo', 'Porto Alegre'],
  'title': 'Localidade',
  'type': 'string'}}

In [13]:
tool_temp.description

'Faz busca online de temperatura de uma localidade'

In [16]:
tool_temp.invoke({"localidade": "Porto Alegre"})

'25ºC'